# Notebook 05: Optuna Hyperparameter Search for SCRUB and TarUn/UNSIR (`05_optuna_scrub_unsir.ipynb`)

Same data / checkpoint / evaluation setup as `03_standard_mu.ipynb`, but only for **SCRUB** and **TarUn (UNSIR)**, one model seed, and the hyperparameters are searched with **Optuna** instead of being fixed.

### Objective (no MIA, accuracy only)
Following Deep Unlearn (Cadet et al., arXiv:2410.01276, Sec. 5.4) but keeping only the accuracy terms on the sets that matter here:

`loss = |RA(f_U) - RA(f_oracle)| + |FA(f_U) - FA(f_oracle)|`

- `RA` = output accuracy on the retain set D_R, `FA` = output accuracy on the forget set D_F.
- The oracle values come from `02_oracle_retrain.ipynb` (same model, same seed). Lower is better; 0 means the unlearned model matches the retrained oracle on both sets.
- D_T (test) and D_V (val) are **not** used to select hyperparameters; they are only reported for the best configuration.

### Searched hyperparameters
- **SCRUB** (SGD): `lr`, `momentum`, `weight_decay`, `epochs`, `msteps`, `kd_T`, `alpha`, `gamma`, retain/forget batch size.
- **UNSIR** (Adam, so there is no momentum): `noise_lr`, `noise_epochs`, `noise_steps`, `impair_lr`, `impair_epochs`, `impair_batches`, `repair_lr`, `repair_epochs`.

Studies are stored in SQLite (`artifacts/optuna/`), so re-running the notebook resumes instead of restarting.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import glob
import time
import subprocess
import sys
import torch
from torch.utils.data import DataLoader, Subset

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    compute_output_accuracy,
    MODEL_CONFIGS
)

from unlearning_methods import (
    unlearn_scrub,
    unlearn_tarun_unsir
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device} | optuna {optuna.__version__}")

## 1. Configuration, Data & Oracle Reference

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m
SEED = 0          # Single model seed used for the whole search

N_TRIALS = {"scrub": 30, "tarun_unsir": 30}   # trials per method
TIMEOUT_SEC = None                            # optional wall-clock limit per study (seconds)

cfg = MODEL_CONFIGS[DATASET_NAME]
assert SEED in cfg["model_seeds"], f"SEED {SEED} not in {cfg['model_seeds']}"
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)
os.makedirs("./artifacts/optuna", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split file not found at {kaggle_split} or {local_split}. Run 01_train_full.ipynb first.")
print(f"Using split artifact: {split_file}")

with open(split_file, "r") as f:
    split_info = json.load(f)

# Construct DataLoaders (identical to 03_standard_mu.ipynb)
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# Original checkpoint (theta_o) for this seed
kaggle_ckpt = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{SEED}.pt"
local_ckpt = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{SEED}.pt"
ckpt_file = kaggle_ckpt if os.path.exists(kaggle_ckpt) else local_ckpt
if not os.path.exists(ckpt_file):
    raise FileNotFoundError(f"Required original checkpoint not found at {kaggle_ckpt} or {local_ckpt}. Run 01_train_full.ipynb first.")
print(f"Using original model checkpoint: {ckpt_file}")

# Oracle reference accuracies (from 02_oracle_retrain.ipynb, same model & seed)
def load_oracle_reference(model_name, seed, search_dirs=("./artifacts/metrics", "/kaggle/input")):
    fname = f"oracle_{model_name}_seed_{seed}.json"
    for d in search_dirs:
        direct = os.path.join(d, fname)
        if os.path.exists(direct):
            path = direct
        else:
            hits = glob.glob(os.path.join(d, "**", fname), recursive=True) if os.path.isdir(d) else []
            if not hits:
                continue
            path = hits[0]
        with open(path, "r") as f:
            r = json.load(f)
        print(f"Using oracle reference: {path}")
        return r["test_metrics"]["output_retain"], r["test_metrics"]["output_forget"]
    raise FileNotFoundError(f"Oracle metrics '{fname}' not found in {search_dirs}. Run 02_oracle_retrain.ipynb first.")

ORACLE_RA, ORACLE_FA = load_oracle_reference(MODEL_NAME, SEED)
print(f"Oracle reference | Retain Acc (RA): {ORACLE_RA:.4f} | Forget Acc (FA): {ORACLE_FA:.4f}")

## 2. Search Spaces & Objective

`loss = |RA - RA_oracle| + |FA - FA_oracle|`, minimized by Optuna's TPE sampler. Every trial starts from the same original checkpoint and the same torch seed, so differences between trials come from the hyperparameters only.

Edit the ranges in `suggest_scrub` / `suggest_unsir` if you want a narrower or wider search.

In [ ]:
def load_original():
    model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt_file, map_location=device))
    return model


def suggest_scrub(trial):
    return dict(
        optim_name="sgd",
        lr=trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        momentum=trial.suggest_float("momentum", 0.0, 0.99),
        weight_decay=trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        epochs=trial.suggest_int("epochs", 5, 20),
        msteps=trial.suggest_int("msteps", 1, 5),
        kd_T=trial.suggest_float("kd_T", 1.0, 8.0),
        alpha=trial.suggest_float("alpha", 0.0, 1.0),
        gamma=trial.suggest_float("gamma", 0.5, 2.0),
        batch_size=trial.suggest_categorical("batch_size", [32, 64, 128]),
        forget_batch_size=trial.suggest_categorical("forget_batch_size", [32, 64, 128]),
    )


def suggest_unsir(trial):
    return dict(
        noise_lr=trial.suggest_float("noise_lr", 1e-2, 1.0, log=True),
        noise_epochs=trial.suggest_int("noise_epochs", 1, 5),
        noise_steps=trial.suggest_int("noise_steps", 2, 16),
        impair_lr=trial.suggest_float("impair_lr", 1e-5, 1e-2, log=True),
        impair_epochs=trial.suggest_int("impair_epochs", 1, 3),
        impair_batches=trial.suggest_int("impair_batches", 1, 20),
        repair_lr=trial.suggest_float("repair_lr", 1e-4, 1e-1, log=True),
        repair_epochs=trial.suggest_int("repair_epochs", 1, 3),
    )


SUGGEST = {"scrub": suggest_scrub, "tarun_unsir": suggest_unsir}


def run_method(method, model, params):
    if method == "scrub":
        return unlearn_scrub(model, train_r_loader, train_f_loader, device=device, **params)
    if method == "tarun_unsir":
        return unlearn_tarun_unsir(model, train_r_loader, forget_loader=train_f_loader,
                                   num_classes=NUM_CLASSES, device=device, **params)
    raise ValueError(method)


def make_objective(method):
    def objective(trial):
        params = SUGGEST[method](trial)
        torch.manual_seed(SEED)
        model = load_original()
        t0 = time.time()
        unlearned = run_method(method, model, params)
        runtime = time.time() - t0

        ra = compute_output_accuracy(unlearned, retain_eval_loader, device=device)
        fa = compute_output_accuracy(unlearned, forget_eval_loader, device=device)
        trial.set_user_attr("retain_acc", ra)
        trial.set_user_attr("forget_acc", fa)
        trial.set_user_attr("runtime_sec", runtime)
        print(f"[{method}] trial {trial.number}: RA={ra:.4f} FA={fa:.4f} "
              f"(oracle {ORACLE_RA:.4f}/{ORACLE_FA:.4f}) loss={abs(ra - ORACLE_RA) + abs(fa - ORACLE_FA):.4f} | {runtime:.0f}s")
        return abs(ra - ORACLE_RA) + abs(fa - ORACLE_FA)
    return objective


def run_study(method):
    study = optuna.create_study(
        study_name=f"{method}_{DATASET_NAME}_{MODEL_NAME}_seed{SEED}",
        storage=f"sqlite:///artifacts/optuna/{method}_{DATASET_NAME}_{MODEL_NAME}_seed{SEED}.db",
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = max(N_TRIALS[method] - done, 0)
    print(f"[{method}] {done} completed trials found, running {remaining} more")
    # RuntimeError (e.g. CUDA OOM) marks only that trial as failed and the search continues
    study.optimize(make_objective(method), n_trials=remaining, timeout=TIMEOUT_SEC, catch=(RuntimeError,))
    return study

## 3. Run the Searches

In [ ]:
study_scrub = run_study("scrub")
print("\nBest SCRUB loss:", study_scrub.best_value)
print("Best SCRUB params:", json.dumps(study_scrub.best_params, indent=2))

In [ ]:
study_unsir = run_study("tarun_unsir")
print("\nBest UNSIR loss:", study_unsir.best_value)
print("Best UNSIR params:", json.dumps(study_unsir.best_params, indent=2))

## 4. Best Configuration: Re-run & Report

Re-runs each method once with its best hyperparameters and reports retain / forget / validation / test accuracy next to the oracle. Validation and test accuracy are reported only; they were not used for selection.

In [ ]:
import pandas as pd

studies = {"scrub": study_scrub, "tarun_unsir": study_unsir}
oracle_val = oracle_test = None
rows = []

for method, study in studies.items():
    best = dict(study.best_params)
    if method == "scrub":
        best["optim_name"] = "sgd"   # fixed, not searched

    torch.manual_seed(SEED)
    model = load_original()
    t0 = time.time()
    unlearned = run_method(method, model, best)
    runtime = time.time() - t0

    ra = compute_output_accuracy(unlearned, retain_eval_loader, device=device)
    fa = compute_output_accuracy(unlearned, forget_eval_loader, device=device)
    va = compute_output_accuracy(unlearned, val_loader, device=device)
    ta = compute_output_accuracy(unlearned, test_loader, device=device)

    res = {
        "method": method, "dataset": DATASET_NAME, "model_name": MODEL_NAME, "seed": SEED, "split_seed": SPLIT_SEED,
        "n_trials_completed": len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        "best_trial": study.best_trial.number, "best_loss": study.best_value, "best_params": best,
        "retain_acc": ra, "forget_acc": fa, "val_acc": va, "test_acc": ta,
        "oracle_retain_acc": ORACLE_RA, "oracle_forget_acc": ORACLE_FA,
        "runtime_sec": runtime,
    }
    path = f"./artifacts/metrics/optuna_{method}_{MODEL_NAME}_seed_{SEED}.json"
    with open(path, "w") as f:
        json.dump(res, f, indent=2)
    print(f"[{method}] saved {path}")
    rows.append({"method": method, "RA": ra, "FA": fa, "VA": va, "TA": ta,
                 "|RA-RA_o|": abs(ra - ORACLE_RA), "|FA-FA_o|": abs(fa - ORACLE_FA),
                 "loss": abs(ra - ORACLE_RA) + abs(fa - ORACLE_FA), "runtime_s": runtime})

rows.append({"method": "oracle (reference)", "RA": ORACLE_RA, "FA": ORACLE_FA, "VA": float("nan"), "TA": float("nan"),
             "|RA-RA_o|": 0.0, "|FA-FA_o|": 0.0, "loss": 0.0, "runtime_s": float("nan")})
summary = pd.DataFrame(rows).set_index("method")
summary.to_csv(f"./artifacts/metrics/summary_optuna_{MODEL_NAME}_seed_{SEED}.csv")
print(summary.round(4).to_string())

In [ ]:
# Top-5 trials per method (lowest loss) with their RA / FA
for method, study in studies.items():
    df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
    df = df[df["state"] == "COMPLETE"].sort_values("value").head(5)
    print(f"\n=== {method}: top 5 trials ===")
    print(df.round(5).to_string(index=False))